# Citation Diagnostic Engine — quickstart

Diagnose the citations of one PubMed Central paper, end to end, and read the
records that come out.

**You need:** an NCBI API key (free, from an NCBI account — it raises the rate
limit from 3 to 10 requests/second and Band 1 is rate-limited without it) and
an Anthropic API key.

**Total runtime:** roughly 15–30 minutes, most of it in Band 2, which makes one
model call per atomic claim. Each cell below states its own expected time.

**Cost:** roughly \$0.20–\$1.00 for a typical reference list. That is an
estimate computed from the per-call token price, *not* a measurement of your
run — the run reports what it actually spent at the end.

The notebook is restart-safe: re-running a cell will not duplicate work or
corrupt the checkpoint.

## 1. Install

*~90 s.*

In [ ]:
import os, pathlib

# The repository is public, so this clones without a token. Point REPO_URL at a
# fork if you have one. If you are already running inside a checkout, the branch
# below finds it and skips the clone.
REPO_URL = 'https://github.com/astonliu/citation-diagnostic-engine.git'
REPO = pathlib.Path('/content/cde')

if pathlib.Path('cde').is_dir() and pathlib.Path('requirements.txt').exists():
    REPO = pathlib.Path.cwd()          # already inside a checkout
elif REPO.exists():
    pass                                # cloned by an earlier run of this cell
else:
    !git clone --depth 1 {REPO_URL} {REPO}

os.chdir(REPO)
!pip install -r requirements.txt -q
print('repo:', REPO)


## 2. Keys

Read from Colab secrets if present, otherwise paste them below. Nothing here
prints a key.

*~5 s.*

In [ ]:
import os

def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

# Paste here ONLY if you are not using Colab secrets.
NCBI_API_KEY = _secret('NCBI_API_KEY') or ''
ANTHROPIC_API_KEY = _secret('ANTHROPIC_API_KEY') or ''

missing = [n for n, v in (('NCBI_API_KEY', NCBI_API_KEY),
                          ('ANTHROPIC_API_KEY', ANTHROPIC_API_KEY)) if not v]
if missing:
    raise SystemExit(
        f'missing {missing}. Add them under the key icon in the Colab sidebar '
        f'(Secrets), or assign them in the cell above. Band 1 needs the NCBI '
        f'key to run at a usable rate; Band 2 needs the Anthropic key.')

os.environ['NCBI_API_KEY'] = NCBI_API_KEY
os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY
print('both keys present')   # never printed, only confirmed

## 3. Smoke test

Prove the install works *before* spending an API call on it.

*~30 s.*

In [ ]:
import cde, cde.refs, cde.claims, cde.diagnose, cde.runtime

print('cde        ', cde.__file__)
for name in ('refs', 'claims', 'diagnose', 'runtime'):
    print(f'cde.{name:9s}', getattr(cde, name, None) or
          __import__(f'cde.{name}', fromlist=['x']).__file__)

# Four fast offline tests: the decision hierarchy, the abstention paths, the
# frozen prompt substrate, and the pinned thresholds. No network, no model.
!python -m pytest -q tests/characterization/test_hierarchy.py \
    tests/characterization/test_constants_and_freeze.py 2>&1 | tail -3

## 4. What each category looks like

Before spending anything, walk the taxonomy: one worked case per category,
each run through the **real** deciding code — `refs.decide.decide` for the
database-resolvable categories, `diagnose.engine.decide_judgment` for the
judgment band. Offline, free, and about a second.

Most of the cases are real published papers. Two cannot be, and the reason is
printed with them rather than hidden: a genuine F1 would mean naming a real
paper as invented, and a genuine F2 would mean asserting that identifiable
authors miscited. **F4 has no real case in this repository yet** — its test is
deliberately left failing so the gap is impossible to miss.

*~5 s, no API calls.*

In [ ]:
!python -m pytest -q tests/characterization/test_taxonomy_cases.py \
    -v --no-header 2>&1 | sed -n '/collected/,$p' | head -20

Now read the cases themselves. Each docstring states the input, the label it
must produce, and why that label rather than the neighbouring one — the F3/F6,
F4/F6 and F5/F8 boundaries are where a taxonomy this fine actually earns its
keep.

*~2 s.*

## 5. Fetch one paper

`PMC4322534` is a 2015 review with a real reference list, and it is one of the
repo's own fixtures, so this cell is reproducible. Change it to any PMC Open
Access id.

*~20 s.*

In [ ]:
import inspect, re
from tests.characterization import test_taxonomy_cases as cases

for name, fn in sorted(vars(cases).items()):
    if not name.startswith('test_') or not callable(fn):
        continue
    doc = inspect.getdoc(fn) or ''
    label = re.match(r'\*\*(F\d|ACCURATE)[^*]*\*\*', doc)
    head = label.group(0).strip('*') if label else name
    print('=' * 78)
    print(head)
    print('-' * 78)
    print(doc)
    print()

In [ ]:
import pathlib, subprocess

PMCID = 'PMC4322534'
XML_DIR = pathlib.Path('data/raw'); XML_DIR.mkdir(parents=True, exist_ok=True)
target = XML_DIR / f'{PMCID}.xml'

if target.exists() and target.stat().st_size > 0:
    print(f'{PMCID} already fetched ({target.stat().st_size:,} bytes)')
else:
    (pathlib.Path('/tmp/ids.txt')).write_text(PMCID + '\n')
    subprocess.run(['bash', 'script/download-data.sh', '/tmp/ids.txt',
                    str(XML_DIR)], check=True)

from cde.refs.parser import parse_pmc_xml
refs = parse_pmc_xml(str(target), source_pmcid=PMCID)
cited = sum(1 for r in refs if (r.citance or '').strip())
print(f'references parsed : {len(refs)}')
print(f'with a citance    : {cited}   <- only these can reach Band 2')

## 6. Band 1 — is each reference the work it claims to be?

Deterministic: identifier resolution plus a title search across PubMed,
Crossref and OpenAlex. No judgment, and no model call except the formatting
filter.

*~2–4 min for a typical reference list — NCBI rate-limited, and the wall time
is almost entirely waiting on it.*

In [ ]:
import json, pathlib, time

OUT = pathlib.Path('results/quickstart'); OUT.mkdir(parents=True, exist_ok=True)
band1_log = OUT / 'band1_lossless_log.jsonl'

# Restart-safe: Band 1 is the expensive-in-wall-time half, so a completed run
# is reused rather than repeated.
if band1_log.exists() and band1_log.stat().st_size > 0:
    print('Band 1 already done; reusing', band1_log)
else:
    t0 = time.time()
    !python -m cde.runtime.cli band1 \
        --xml-dir {XML_DIR} --out-dir {OUT} --model claude-haiku-4-5
    print(f'Band 1 took {time.time() - t0:.0f}s')

import collections
rows = [json.loads(l) for l in band1_log.read_text().splitlines() if l]
counts = collections.Counter(r.get('label') or '-' for r in rows)
print('\nBand 1 disposition')
for label, n in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0])):
    print(f'  {label:16s} {n:4d}')
print(f'  {"total":16s} {len(rows):4d}')
print('\nOnly `cleared` references go on to Band 2.')

## 7. Band 2 — is the citing sentence a fair use of that work?

One model call per atomic claim, so this is where the time and the money go.
A live progress line and a per-document checkpoint mean a disconnect costs you
the current document, not the run.

*~10–25 min and roughly \$0.20–\$1.00, depending on how many references
cleared. The dollar figure is an estimate from the per-call price; the actual
spend is reported at the end from the token ledger.*

In [ ]:
import pathlib, time

judgment = OUT / 'judgment_predictions.jsonl'

# run_natural_judgment checkpoints per document and resumes from its own hash
# chain, so re-running this cell continues rather than restarting -- and never
# appends a second copy of a document already in the chain.
t0 = time.time()
!python -m cde.runtime.cli band2 \
    --xml-dir {XML_DIR} --out-dir {OUT} --model claude-haiku-4-5
print(f'\nBand 2 took {(time.time() - t0) / 60:.1f} min')

manifest_path = OUT / 'run_manifest.json'
if manifest_path.exists():
    m = json.loads(manifest_path.read_text())
    spend = (m.get('token_usage') or {}).get('total', {})
    print('actual spend :', spend.get('cost_usd', 'not_collected'))
    print('model calls  :', spend.get('calls', '-'))

## 8. Read the output

One record per citation pair, with the claims it was decomposed into and the
evidence spans each verdict rests on.

*~5 s.*

In [ ]:
import json, pandas as pd

records = [json.loads(l) for l in judgment.read_text().splitlines() if l]
df = pd.DataFrame(records)

print('route distribution -- the denominator every rate divides by')
print(df['route'].fillna('(excluded)').value_counts().to_string())
print('\ndisposition')
print(df['disposition'].value_counts().to_string())

flagged = [r for r in records if r.get('findings')]
print(f'\n{len(flagged)} flagged pair(s) of {len(records)}\n')
for r in flagged:
    print('=' * 78)
    print(f"{r['citation_id']}   {','.join(r['findings'])}"
          f"   ({r.get('terminal_reason', '')})")
    print(f"citing: {r.get('citing_sentence', '')}")
    for claim in (r.get('atomic_claims') or []):
        print(f'  claim: {claim}')
    for v in (r.get('coverage_verdicts') or []):
        span = (v.get('evidence_span') or '').strip()
        print(f"  established={v.get('established')!r}"
              + (f'\n    span: {span[:160]}' if span else ''))

## Where to go next, and where the limits are

**Your own paper.** Change `PMCID` in cell 4 to any PMC Open Access id and
re-run from there. To do several, put one id per line in the ids file that
`script/download-data.sh` reads.

**This is the demonstration path, not the governed one.** `cde.runtime.cli`
deliberately skips the checks that make a run's numbers reportable — the
corpus manifest, the model allowlist, the tree-integrity check, the
different-family judge. The records it writes are real; the population they
were computed over is not governed, so do not compute a published rate from
this output. `cde.runtime.production_launcher` is the entry point that
enforces all of it.

**The limits that will show up in your results:**

- **Evidence is PMC-only,** and abstract-scoped unless PMC has the full text. A
  claim supported in a paper's Results but absent from its abstract reads as
  unestablished. Each record stamps the scope it was judged at.
- **F4** (overstatement) has a development mode that is not corpus-calibrated.
- **F5** (superseded) ships escalation-only: it surfaces the contradiction and
  flags it, and never proposes a replacement in this build.
- **F7** (wrong entity) is pending an advisor lock on its entity authorities.
- A **hold** is not a negative. `UNJUDGEABLE` means the evidence did not settle
  the question, and counting it either way will bias a rate — see
  `doc/evaluation.md`.

Full documentation: [`doc/pipeline.md`](../doc/pipeline.md),
[`doc/taxonomy.md`](../doc/taxonomy.md), [`doc/evaluation.md`](../doc/evaluation.md).